# Controlled interaction experiments — Pilot Model v5

Проверяем явные взаимодействия: они могут позволить деревьям различать эффект температуры или выходного по маршруту и часу. Цель — `boardings` для `(route, date, hour)`. Контроль — tuned XGBoost + C+W + weighted D, half-life 180 дней.

Каждый кандидат добавляется отдельно к чистому v5: иначе нельзя приписать разницу конкретному признаку. Все признаки сразу не добавляем, чтобы не смешать эффекты. Временные folds Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct остаются неизменными для честного сравнения. Nov–Dec leaderboard не используется для выбора: его target недоступен в исторической валидации, а повторный подбор по leaderboard переобучит решение на финальном горизонте.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/src/interaction_features_experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(ROOT / 'ml/src'))
from interaction_features_experiment import run_experiment
result = run_experiment()
names = [fold['name'] for fold in result['folds']]


## 1. Pilot v5 reference

Контроль должен точно воспроизвести сохранённые абсолютные ошибки каждого fold и маршрута; при расхождении выполнение прерывается.

In [ ]:
def show_configuration(name):
    folds = result['fold_results'][name]
    summary = result['summaries'][name]
    comparison = result['comparisons_vs_pilot_v5'].get(name, {})
    display(pd.DataFrame([{'fold': fold, 'WAPE': metrics['wape'], 'AE': metrics['absolute_error'], 'fit_s': metrics['train_seconds'], 'predict_s': metrics['inference_seconds']} for fold, metrics in zip(names, folds)]).round(6))
    print('Pooled WAPE:', round(summary['pooled_wape'], 6), 'WAPE-score:', round(summary['wape_score'], 6), 'Pooled AE:', summary['pooled_absolute_error'], 'Decision:', comparison.get('decision', 'REFERENCE'))
    if comparison:
        print('Delta pooled WAPE:', round(comparison['delta_pooled_wape'], 6), 'Fold wins:', comparison['fold_wins'])
        display(pd.DataFrame([{'route': route, 'delta_WAPE': delta} for route, delta in comparison['route_delta_pooled_wape'].items()]).sort_values('delta_WAPE').round(6))
show_configuration('Pilot v5')


## 2. Route × temperature bucket

Добавлен только `route_temperature_bucket`; исходные route и temperature_bucket сохранены.

In [ ]:
show_configuration('route_temperature_bucket')


## 3. Route × day off

Возвращаемся к чистому v5 и добавляем только `route_is_day_off`.

In [ ]:
show_configuration('route_is_day_off')


## 4. Hour × temperature bucket

Снова сравниваем с чистым v5; добавлен только `hour_temperature_bucket`.

In [ ]:
show_configuration('hour_temperature_bucket')


## 5. Combined check при двух или более SUPPORTED

В combined вариант включаются только SUPPORTED взаимодействия, без нового HPO.

In [ ]:
if 'combined_supported' in result['summaries']:
    show_configuration('combined_supported')
else:
    print('Combined check skipped: fewer than two SUPPORTED candidates.')


## 6. Итог

Основная метрика — pooled WAPE; отрицательная Δ означает улучшение. Route 5 исключён из route WAPE, поскольку его target равен нулю.

In [ ]:
rows = []
for name, summary in result['summaries'].items():
    comparison = result['comparisons_vs_pilot_v5'].get(name, {})
    rows.append({'Configuration': name, **{fold: metrics['wape'] for fold, metrics in zip(names, result['fold_results'][name])}, 'Pooled WAPE': summary['pooled_wape'], 'Δ vs Pilot v5': comparison.get('delta_pooled_wape', 0.0), 'Pooled AE': summary['pooled_absolute_error'], 'Decision': comparison.get('decision', 'REFERENCE')})
display(pd.DataFrame(rows).round(6))
